# contrastive_finetune: contrastive fine-tuning on top of the distilled main-line student (FineWeb QA pairs)

**Start:** the current main line, `main_v3dom_s21600` (1k base, BGE-M3 + instructed 8K-embed, mix v3 + domain texts, 21,600 steps, lr 1e-4; TR-MTEB **68.27**, seed 43: 68.30).

**Training:** in-batch-negative InfoNCE on 300,000 (search query, web document) pairs of the advisor's synthetic FineWeb QA data (`fineweb_qa/sub/pairs_query_train_300k.jsonl`; decontaminated against TR-MTEB, Mizan and the two RAG sets by `prepare_synth_qa`; documents cut at 2,500 characters), one epoch (~2,340 steps, ~30-40 min on an A100, not measured), with the **anchor** (cosine 10 / Gram 200 toward a frozen copy of the starting model) and lr 1e-5: the same settings as the stage-1 anchor run. The in-run validation uses the stage-1 `pairs_val.jsonl` (937 pairs) so the MRR is comparable with the earlier contrastive runs; it is a monitor, not a selection criterion.

**Why:** the one untried combination. After-distillation contrastive on the stage-1 QA pairs (38K Wikipedia QA pairs): plain 67.11 (forgetting), anchor 67.74, soup 67.80 against 67.71, i.e. neutral; FineWeb pairs inside distillation (joint): 67.06 / 67.01 against 67.16. FineWeb is ~8x larger and a different genre (web/news). Expected effect small (at most ~+0.1); risks seen before: topic classification (−0.8 to −2.0 with FineWeb pairs in joint training).

**Outputs:** the contrastive model and a 50/50 weight soup with the starting model (`SHARES = [0.5]`; the best stage-1 result was a soup). Evaluate both with eval_trio against 68.27 (seed 42, the same starting weights); read only differences above 0.3; look at retrieval, topic classification and STS. Re-running after a disconnect resumes the unfinished run and skips finished ones; the last cell releases the runtime. Delete this notebook when the results are in.


In [ ]:
import os
from google.colab import drive, userdata

drive.mount('/content/drive')

REPO_URL = 'https://github.com/mericberktas/bitirme.git'
BRANCH = 'main'
DRIVE_ROOT = '/content/drive/MyDrive/bitirme'
REPO_DIR = '/content/repo'

os.makedirs(DRIVE_ROOT, exist_ok=True)
# Models are downloaded to the LOCAL disk each session. Do not put the HF cache on Drive:
# symlinks and large files are not stored reliably there (it produced corrupted weights).
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} fetch --quiet && git -C {REPO_DIR} checkout --quiet {BRANCH} && git -C {REPO_DIR} pull --quiet
else:
    !git clone --quiet --branch {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install --quiet -e .
!git log -1 --oneline

In [ ]:
# --- Make sure the environment is usable (safe to re-run; covers a restarted runtime) ---
import importlib.util, os, subprocess, sys

DRIVE_ROOT = globals().get('DRIVE_ROOT', '/content/drive/MyDrive/bitirme')
REPO_DIR = globals().get('REPO_DIR', '/content/repo')

if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
if not os.path.isdir(f'{REPO_DIR}/src'):
    raise RuntimeError('Repo not found: run the Setup cell (section 1) first.')
os.chdir(REPO_DIR)  # config paths below are relative to the repo root

if importlib.util.find_spec('turkish_distill') is None:
    print('turkish_distill is not installed in this runtime: installing it now...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-e', REPO_DIR], check=True)
    importlib.invalidate_caches()
    if importlib.util.find_spec('turkish_distill') is None:
        sys.path.insert(0, f'{REPO_DIR}/src')
        os.environ['PYTHONPATH'] = f'{REPO_DIR}/src' + os.pathsep + os.environ.get('PYTHONPATH', '')

os.environ.setdefault('HF_HOME', '/content/hf_cache')  # local disk, never Drive
if 'HF_TOKEN' not in os.environ:
    try:
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception as exc:
        print('HF_TOKEN not set (only needed for gated models):', exc)



In [ ]:
# Contrastive run: (name, extra overrides). A later override wins.
DISTILLED = f'{DRIVE_ROOT}/runs/main_v3dom_s21600/20261120-143553-9fa898c/final'   # TR-MTEB 68.27
CONFIG = 'configs/contrastive_after_distill.yaml'
PAIRS = f'{DRIVE_ROOT}/data/fineweb_qa/sub/pairs_query_train_300k.jsonl'
BASE = ['train.lr=1e-5', 'loss.anchor_cosine_weight=10', 'loss.anchor_gram_weight=200', f'data.pairs_train={PAIRS}',
        'train.eval_every=300', 'train.ckpt_every=300']
RUNS = [
    ('ctr_fw_anchor_lr1e5', BASE),
]
SHARES = [0.5]                            # share of the contrastive model in a soup with the distilled model

import glob, os

assert os.path.isdir(DISTILLED), f'distilled model not found: {DISTILLED}'
assert os.path.exists(PAIRS), f'pairs not found: {PAIRS}'
finals = {}
for name, extra in RUNS:
    run_dirs = sorted(glob.glob(f'{DRIVE_ROOT}/runs/{name}/*/'))
    done = [d for d in run_dirs if os.path.isdir(d + 'final')]
    if done:
        print();print(f'########## {name}: already finished, skipped')
        finals[name] = done[-1] + 'final'
        continue
    run_id_arg = f'--run-id {os.path.basename(run_dirs[-1].rstrip("/"))}' if run_dirs else ''   # resume after a disconnect
    overrides = ' '.join(f'--override {o}' for o in [f'name={name}', f'student.hf_id={DISTILLED}'] + extra)
    print();print(f'########## {name} {run_id_arg or "(new run)"}')
    get_ipython().system(f'python -m turkish_distill.train_contrastive --config {CONFIG} {run_id_arg} {overrides}')
    run_dirs = sorted(glob.glob(f'{DRIVE_ROOT}/runs/{name}/*/'))
    finals[name] = run_dirs[-1] + 'final'
print(finals)


In [ ]:
# Soups: distilled model blended with each contrastive model; saved to Drive/models/ (evaluate with eval_trio presets).
from turkish_distill.soup import soup_models

for name, final in finals.items():
    for share in SHARES:
        out = f'{DRIVE_ROOT}/models/soup_{name}_c{int(round(share * 100))}'
        if os.path.exists(out + '/SOUP.json'):
            print('exists, skipped:', out)
            continue
        soup_models([DISTILLED, final], [1 - share, share], out, label=f'distilled + {name}')
        print('saved:', out)


In [ ]:
# Paste these into eval_trio's PRESETS (they are written out here so the paths need no guessing):
for name, final in finals.items():
    print(f"    '{name}': dict(model='{final}'),")
    for share in SHARES:
        print(f"    'soup_{name}_c{int(round(share * 100))}': dict(model=f'{{DRIVE_ROOT}}/models/soup_{name}_c{int(round(share * 100))}'),")


In [ ]:
from google.colab import runtime
runtime.unassign()
